# 🚀 CompX Orbit — 100% Free AI Voice Enhancer Server (DeepFilterNet + Resemble Enhance)
### Runs on Google Colab's Free NVIDIA Tesla T4 GPU with Cloudflare Tunnel (Zero Cost!)

**Instructions:**
1. Go to **Runtime > Change runtime type** -> Select **T4 GPU**.
2. Click the **Play button** below to run the cell.
3. Copy the generated `https://...trycloudflare.com` URL and paste it into your Next.js Vercel environment as `AI_VOICE_ENHANCE_URL`!

In [ ]:
# 1. Clean previous runs & install ffmpeg
!pkill -f uvicorn 2>/dev/null || true
!pkill -f cloudflared 2>/dev/null || true
!apt-get install -y -qq ffmpeg

# 2. Install FastAPI and AI Voice Engines
print("📦 Installing FastAPI, Resemble Enhance, & dependencies...")
!pip install -q fastapi uvicorn python-multipart torch torchaudio
!pip install -q git+https://github.com/resemble-ai/resemble-enhance.git

# 3. Download cloudflared for free instant public HTTPS tunnel (No account required!)
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared

# 4. Write FastAPI backend app
app_code = '''
import os, subprocess, tempfile, threading, shutil
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware
from starlette.background import BackgroundTask

app = FastAPI(title="Orbit AI Voice Enhancer")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

lock = threading.Lock()
_df = None

def to_wav(src, dst, sr=48000):
    r = subprocess.run(["ffmpeg", "-y", "-i", src, "-vn", "-ac", "1", "-ar", str(sr), dst], capture_output=True)
    if r.returncode != 0:
        raise HTTPException(400, "Could not decode audio with ffmpeg")

def deepfilter(in_wav, out_wav, atten_db):
    global _df
    try:
        from df.enhance import enhance, init_df, load_audio, save_audio
        if _df is None: _df = init_df()
        model, state, _ = _df
        audio, _ = load_audio(in_wav, sr=state.sr())
        limit = None if atten_db >= 100 else float(atten_db)
        out = enhance(model, state, audio, atten_lim_db=limit)
        save_audio(out_wav, out, state.sr())
        return True
    except Exception as e:
        print("DeepFilterNet fallback to Resemble Denoise:", e)
        return False

def resemble(in_wav, out_wav, mode, denoise_strength):
    import torch, torchaudio
    from resemble_enhance.enhancer.inference import denoise, enhance
    device = "cuda" if torch.cuda.is_available() else "cpu"
    wav, sr = torchaudio.load(in_wav)
    wav = wav.mean(dim=0)
    if mode == "denoise":
        out, new_sr = denoise(wav, sr, device)
    elif mode == "enhance":
        out, new_sr = enhance(wav, sr, device, nfe=64, solver="midpoint", lambd=float(denoise_strength), tau=0.5)
    else: # both
        denoised, d_sr = denoise(wav, sr, device)
        out, new_sr = enhance(denoised, d_sr, device, nfe=64, solver="midpoint", lambd=float(denoise_strength), tau=0.5)
    torchaudio.save(out_wav, out.unsqueeze(0).cpu(), new_sr)

@app.get("/health")
def health():
    import torch
    return {"status": "online", "gpu": torch.cuda.is_available(), "name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}

@app.post("/process")
def process(file: UploadFile = File(...), mode: str = Form("both"), atten_db: float = Form(100.0), denoise_strength: float = Form(0.8)):
    tmp = tempfile.mkdtemp()
    cleanup = BackgroundTask(shutil.rmtree, tmp, ignore_errors=True)
    try:
        raw = os.path.join(tmp, "raw_" + (file.filename or "audio"))
        with open(raw, "wb") as f: shutil.copyfileobj(file.file, f)
        cur = os.path.join(tmp, "cur.wav")
        to_wav(raw, cur, 48000)
        with lock:
            if mode == "denoise":
                nxt = os.path.join(tmp, "denoised.wav")
                if not deepfilter(cur, nxt, atten_db):
                    resemble(cur, nxt, "denoise", denoise_strength)
                cur = nxt
            elif mode == "enhance":
                nxt = os.path.join(tmp, "enhanced.wav")
                resemble(cur, nxt, "enhance", denoise_strength)
                cur = nxt
            else: # both
                nxt = os.path.join(tmp, "denoised.wav")
                if deepfilter(cur, nxt, atten_db):
                    cur = nxt
                    nxt2 = os.path.join(tmp, "enhanced.wav")
                    resemble(cur, nxt2, "enhance", denoise_strength)
                    cur = nxt2
                else:
                    resemble(cur, nxt, "both", denoise_strength)
                    cur = nxt
        return FileResponse(cur, media_type="audio/wav", filename="enhanced.wav", background=cleanup)
    except Exception as e:
        shutil.rmtree(tmp, ignore_errors=True)
        raise HTTPException(500, detail=str(e))
'''
with open('app.py', 'w') as f: f.write(app_code)

# 5. Launch FastAPI background server on port 7860
import subprocess, time, os, re
proc = subprocess.Popen(["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "7860"])
time.sleep(3)

# 6. Start Cloudflare Tunnel (Non-blocking log pipe to avoid buffer deadlocks)
print("\n===================================================")
print("🚀 STARTING CLOUDFLARE PUBLIC HTTPS TUNNEL...")
print("===================================================")
with open("tunnel.log", "w") as log_file:
    tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:7860"], stdout=log_file, stderr=log_file)

url = None
for attempt in range(25):
    time.sleep(1)
    if os.path.exists("tunnel.log"):
        with open("tunnel.log", "r") as f:
            log_content = f.read()
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', log_content)
            if match:
                url = match.group(0)
                break

if url:
    print("\n🎉 SUCCESS! YOUR FREE SERVERLESS AI URL IS:")
    print("👉 " + url)
    print("\nPaste this URL into your Vercel or Next.js .env as: AI_VOICE_ENHANCE_URL")
else:
    print("Tunnel log status:")
    with open("tunnel.log", "r") as f: print(f.read())
